A consumer that reports which partitions it owns. Start several with the same group

In [ ]:
import sys, time
from confluent_kafka import Consumer

In [2]:
name = "a-name"
seconds = 1

In [3]:
def on_assign(consumer, partitions):
    print(f"{name}: assigned partitions: {[p.partition for p in partitions]}", flush=True)

def on_revoke(consumer, partitions):
    print(f"{name}: revoked partitions {[p.partition for p in partitions]}", flush=True)

In [ ]:
consumer = Consumer({"bootstrap.servers": "localhost:9092",
                     "group.id": "group-demo",
                     "auto.offset.reset": "earliest",
                     "session.timeout.ms": 6000})
consumer.subscribe(["orders"], on_assign=on_assign, on_revoke=on_revoke)
counts = {}
end = time.time() + seconds

try:
    while time.time() < end:
        msg = consumer.poll(0.5)
        if msg is not None and not msg.error():
            counts[msg.partition()] = counts.get(msg.partition(), 0) + 1
except KeyboardInterrupt:
    print(f"{name}: stopping", flush=True)
finally:
    consumer.close()
    print(f"{name}: read {counts}, closed", flush=True)

In [ ]:
# Test run
# > papermill group_consumer.ipynb output_group_consumer.ipynb -p name "A" -p seconds 300
# > papermill group_consumer.ipynb output_group_consumer.ipynb -p name "B" -p seconds 300

In [ ]:
# Check the consumer groups
# > kafka-consumer-groups.sh --bootstrap-server $BS --describe --group group-demo --members
"""/opt/kafka/bin $ kafka-consumer-groups.sh --bootstrap-server $BS --describe --group group-demo --members

GROUP           CONSUMER-ID                                  HOST            CLIENT-ID       #PARTITIONS     
group-demo      rdkafka-32925d33-0dd5-47d0-a83a-c307e35f1d22 /172.17.0.1     rdkafka         3"""